# Script de Análise Comparativa para Experimentos de Classificação de Câncer Renal.


Este script carrega os resultados de múltiplos treinamentos de modelos,
gera gráficos comparativos das curvas de aprendizado, e produz relatórios
comparativos de performance no conjunto de teste (Relatório de Classificação,
Matriz de Confusão, Curva ROC).

## IMPORTAÇÕES

In [ ]:
# --- Bibliotecas Padrão do Python ---
import os
import pickle
import random
import time
import platform
import math


# --- Bibliotecas de Terceiros ---
# Análise de Dados e Visualização
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import cv2
from tqdm import tqdm

# Machine Learning e Avaliação (Scikit-learn)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)
from sklearn.preprocessing import label_binarize

# Deep Learning (PyTorch e Torchvision)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torch.optim import lr_scheduler
from torchvision import datasets, models, transforms, utils

In [ ]:
pip freeze > requirements.txt

## CONEXÃO COM O GOOGLE DRIVE

In [ ]:
try:
  import google.colab
  COLAB = True
  print("Rodando no Colab")
except:
  COLAB = False
  print("Rodando Localmente")

if COLAB:
  from google.colab import drive
  drive.mount('/content/drive', timeout_ms=10200000)

## CONFIGURAÇÃO GERAL DOS EXPERIMENTOS

In [ ]:
# --- CONFIGURAÇÃO DO EXPERIMENTO ---
MODELOS_PARA_ANALISAR = ['alexnet', 'resnet50', 'vgg16', 'mobilenet_v2','efficientnet_b7', 'vit_b_16']
TIPOS_DATASET = ['original', 'wavelet']
CLASSES = ['Normal', 'Tumor']

if COLAB:
    BASE_PATH = ""
else:
    BASE_PATH = ""


## DEFINIÇÃO DAS FUNÇÕES


In [ ]:
# --- CÉLULA DE DEFINIÇÃO DAS FUNÇÕES DE ANÁLISE ---

def configurar_caminhos(base_path, modelos, datasets):
    """Cria os diretórios necessários e monta um dicionário com os caminhos para os resultados."""
    print("Configurando caminhos...")
    caminho_treinamentos = os.path.join(base_path, "treinamentos")
    relatorio_dir_agrupado = os.path.join(caminho_treinamentos, "relatorios_agrupados")
    os.makedirs(relatorio_dir_agrupado, exist_ok=True)
    print(f"Relatórios agrupados serão salvos em: {relatorio_dir_agrupado}")

    caminhos_resultados = {}
    for modelo in modelos:
        caminhos_resultados[modelo] = {}
        for tipo_ds in datasets:
            save_dir = os.path.join(caminho_treinamentos, modelo, tipo_ds, "relatorios")
            caminhos_resultados[modelo][tipo_ds] = {
                'training': os.path.join(save_dir, f'metricas_{modelo}_training.pkl'),
                'test': os.path.join(save_dir, f'metricas_{modelo}_test.pkl')
            }
    return caminhos_resultados, relatorio_dir_agrupado

def carregar_dados_experimentos(caminhos_resultados, tipo_dados='training'):
    """Carrega os dados de treinamento ou teste de todos os experimentos."""
    dados_carregados = {}
    print(f"\nCarregando dados de '{tipo_dados}' para análise...")
    for modelo, datasets in caminhos_resultados.items():
        for tipo_ds, caminhos in datasets.items():
            nome_experimento = f"{modelo.replace('_', ' ').title()} ({tipo_ds.capitalize()})"
            caminho_pkl = caminhos[tipo_dados]

            if os.path.exists(caminho_pkl):
                with open(caminho_pkl, "rb") as f:
                    dados_carregados[nome_experimento] = pickle.load(f)
                print(f"  ✔ Dados para '{nome_experimento}' carregados.")
            else:
                print(f"  ✘ AVISO: Arquivo não encontrado para '{nome_experimento}'. Pulando.")
    return dados_carregados

In [ ]:
def plotar_curvas_comparativas(historicos, save_path):
    """
    Plota as curvas de perda e acurácia de validação, com legenda única
    e destaque para o melhor modelo.
    """
    print("\nGerando gráfico comparativo das curvas de aprendizado...")


    melhor_modelo_nome = ''
    maior_acuracia_final = 0
    if historicos:
        for nome, metricas in historicos.items():
            acuracia_final = metricas.get('val_acc', [0])[-1]
            if acuracia_final > maior_acuracia_final:
                maior_acuracia_final = acuracia_final
                melhor_modelo_nome = nome

    num_modelos = len(historicos)
    cmap = plt.get_cmap('tab20')
    cores = cmap.colors
    estilos = ['-', '--', ':', '-.']



    fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

    for i, (nome, metricas) in enumerate(historicos.items()):
        val_loss = metricas.get('val_loss', [])
        val_acc = metricas.get('val_acc', [])
        epochs = range(1, len(val_loss) + 1)

        cor = cores[i % len(cores)]
        estilo = estilos[i % len(estilos)]

        if nome == melhor_modelo_nome:
            largura_linha = 3.0
            ordem_plot = 10
        else:
            largura_linha = 1.5
            ordem_plot = 5

        axes[0].plot(epochs, val_loss, label=nome, marker='.', linestyle=estilo,
                     color=cor, lw=largura_linha, zorder=ordem_plot)

        axes[1].plot(epochs, val_acc, label=nome, marker='.', linestyle=estilo,
                     color=cor, lw=largura_linha, zorder=ordem_plot)


    axes[0].set_title('Validation Loss Over Epochs')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, linestyle='--', alpha=0.6)

    axes[1].set_title('Validation Accuracy Over Epochs')
    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel('Accuracy')
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    axes[1].set_ylim(bottom=0)

    handles, labels = axes[1].get_legend_handles_labels()

    fig.legend(handles, labels,
               loc='lower center',
               bbox_to_anchor=(0.5, -0.05),
               ncol=5,
               fancybox=True,
               shadow=True)

    fig.suptitle('Learning Curve Analysis', fontsize=18)
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])

    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Gráfico comparativo salvo em: {save_path}")

In [ ]:
# --- FUNÇÃO UNIFICADA PARA GERAR RELATÓRIOS ---
def gerar_relatorios_unificados(dados, tipo_relatorio, classes, save_dir):
    """
    Gera um conjunto completo de relatórios comparativos (texto, matriz, ROC)
    para um determinado conjunto de dados (ex: 'Teste' ou 'Validação').
    """
    print(f"\nGerando relatórios comparativos para o conjunto de '{tipo_relatorio}'...")
    num_experimentos = len(dados)

    sufixo_arquivo = tipo_relatorio.lower().replace(' ', '_').replace('(', '').replace(')', '')

    # 1. Relatório de Classificação em Texto (COM MÉTRICAS PRECISAS)
    path_relatorio_txt = os.path.join(save_dir, f"relatorio_classificacao_{sufixo_arquivo}.txt")
    with open(path_relatorio_txt, "w") as f:
        f.write(f"RELATÓRIO DE CLASSIFICAÇÃO COMPARATIVO ({tipo_relatorio})\n")
        for nome, metricas in dados.items():
            f.write("\n" + "="*80 + f"\nResultado para: {nome}\n" + "="*80 + "\n\n")

            acc_precisa = metricas.get('test_acc') or metricas.get('val_acc') # Pega a acurácia correta
            loss_precisa = metricas.get('test_loss') or metricas.get('val_loss') # Pega a perda correta

            if isinstance(acc_precisa, list) and acc_precisa:
              acc_precisa_value = acc_precisa[-1]
            else:
              acc_precisa_value = acc_precisa

            if isinstance(loss_precisa, list) and loss_precisa:
              loss_precisa_value = loss_precisa[-1]
            else:
              loss_precisa_value = loss_precisa


            if acc_precisa_value is not None:
                f.write(f"Acurácia Precisa (Não Arredondada): {acc_precisa_value:.4f} ({acc_precisa_value:.2%})\n")
            if loss_precisa_value is not None:
                f.write(f"Perda Precisa (Não Arredondada):   {loss_precisa_value:.4f}\n\n")

            f.write("--- Relatório Detalhado (Arredondado) ---\n")
            if 'all_labels' in metricas and 'all_preds' in metricas:
                f.write(classification_report(metricas['all_labels'], metricas['all_preds'], target_names=classes))
            else:
                f.write("Dados de predição/labels não encontrados.")
    print(f"Relatório de classificação ({tipo_relatorio}) salvo em: {path_relatorio_txt}")


    # --- 2. Matrizes de Confusão ---

    num_cols = 4
    num_rows = math.ceil(num_experimentos / num_cols)

    fig_cm, axes_cm = plt.subplots(num_rows, num_cols, figsize=(6 * num_cols, 5 * num_rows))
    fig_cm.suptitle(f'Comparison of Confusion Matrices ({tipo_relatorio})', fontsize=16)

    axes_flat = axes_cm.flatten()

    for i, (nome, metricas) in enumerate(dados.items()):
        ax = axes_flat[i]
        if 'all_labels' in metricas and 'all_preds' in metricas:
            cm = confusion_matrix(metricas['all_labels'], metricas['all_preds'])
            disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
            disp.plot(ax=ax, cmap=plt.cm.Blues, colorbar=True)
        ax.set_title(nome)
        ax.grid(False)

    for i in range(num_experimentos, len(axes_flat)):
        axes_flat[i].axis('off')

    path_matriz_cm = os.path.join(save_dir, f"comparativo_matrizes_confusao_{sufixo_arquivo}.png")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(path_matriz_cm)
    plt.close(fig_cm)
    print(f"Matrizes de confusão ({tipo_relatorio}) salvas em: {path_matriz_cm}")

   # 3. Curvas ROC Sobrepostas

    # --- 1. PRÉ-CÁLCULO E ORDENAÇÃO DOS DADOS ---
    print("Calculando e ordenando os resultados dos modelos...")
    for nome, metricas in dados.items():
        if 'all_labels' in metricas and 'all_probs' in metricas and len(classes) == 2 and len(metricas['all_labels']) > 0:
            all_labels = metricas['all_labels']
            all_probs_pos = np.array(metricas['all_probs'])[:, 1]
            fpr, tpr, _ = roc_curve(all_labels, all_probs_pos)

            metricas['fpr'] = fpr
            metricas['tpr'] = tpr
            metricas['roc_auc'] = auc(fpr, tpr)
        else:
            metricas['roc_auc'] = 0.0

    dados_ordenados = dict(sorted(dados.items(), key=lambda item: item[1]['roc_auc'], reverse=True))


    # --- 2. IDENTIFICAR O MELHOR DESEMPENHO E VERIFICAR EMPATES ---
    primeiro_modelo = next(iter(dados_ordenados.values()), None)
    max_auc = primeiro_modelo['roc_auc'] if primeiro_modelo else 0.0

    num_melhores_modelos = sum(1 for m in dados_ordenados.values() if m.get('roc_auc') == max_auc)


    # --- 3. PREPARAÇÃO DOS ESTILOS DE PLOTAGEM ---
    num_modelos = len(dados_ordenados)
    cmap = plt.get_cmap('tab20')
    cores = cmap.colors
    estilos = ['-', '--', ':', '-.']


    # --- 4. CRIAÇÃO DO GRÁFICO ---
    print("Gerando o gráfico comparativo...")
    plt.figure(figsize=(12, 11))
    plt.title(f'Comparison of ROC Curves ({tipo_relatorio})', fontsize=16)

    for i, (nome, metricas) in enumerate(dados_ordenados.items()):
        if 'fpr' not in metricas:
            continue

        fpr = metricas['fpr']
        tpr = metricas['tpr']
        roc_auc = metricas['roc_auc']

        cor = cores[i % len(cores)]
        estilo = estilos[i % len(estilos)]

        if roc_auc == max_auc:
            largura_linha = 3.5
            ordem_plot = 10 #
            if num_melhores_modelos > 1:
                label_sufixo = " (Best Performance)"
            else:
                label_sufixo = " (Best Model)"
        else:
            largura_linha = 2.0
            ordem_plot = 5
            label_sufixo = ""

        plt.plot(fpr, tpr,
                 color=cor,
                 linestyle=estilo,
                 lw=largura_linha,
                 zorder=ordem_plot,
                 label=f'{nome} (AUC = {roc_auc:.3f}){label_sufixo}')

    plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Chance (AUC = 0.5)')

    plt.xlabel('False Positive Rate (FPR)', fontsize=12)
    plt.ylabel('True Positive Rate (TPR)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)


    # --- 5. CONFIGURAÇÃO DA LEGENDA E SALVAMENTO ---
    plt.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.15),
        ncol=4,
        fancybox=True,
        shadow=True,
        fontsize=10
    )

    path_roc = os.path.join(save_dir, f"comparativo_curvas_roc_{sufixo_arquivo}.png")

    plt.savefig(path_roc, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"Curvas ROC ({tipo_relatorio}) salvas em: {path_roc}")

In [ ]:
# ==============================================================================
# FUNÇÕES PARA ANÁLISE DE EFICIÊNCIA
# ==============================================================================

def carregar_arquitetura_modelo(nome_modelo):
    """Carrega a arquitetura de um modelo do torchvision."""
    if nome_modelo == 'alexnet':
        return models.alexnet(weights='DEFAULT')
    elif nome_modelo == 'resnet50':
        return models.resnet50(weights='DEFAULT')
    elif nome_modelo == 'vgg16':
        return models.vgg16(weights='DEFAULT')
    elif nome_modelo == 'mobilenet_v2':
        return models.mobilenet_v2(weights='DEFAULT')
    elif nome_modelo == 'efficientnet_b7':
        return models.efficientnet_b7(weights='DEFAULT')
    elif nome_modelo == 'vit_b_16':
        return models.vit_b_16(weights='DEFAULT')
    else:
        raise ValueError(f"Modelo '{nome_modelo}' não reconhecido.")

def gerar_relatorio_eficiencia(modelos, save_dir):
    """
    Calcula o número de parâmetros e o tempo de inferência para uma lista de modelos
    e salva um relatório em texto.
    """
    print("\nGerando relatório de eficiência (Parâmetros e Tempo de Inferência)...")

    os.makedirs(save_dir, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dummy_input = torch.randn(1, 3, 224, 224).to(device)

    resultados_eficiencia = []

    for nome_modelo in tqdm(modelos, desc="Analisando modelos"):
        model = carregar_arquitetura_modelo(nome_modelo)
        model.to(device)
        model.eval()

        total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

        repetitions = 100
        timings = []

        # Warm-up
        for _ in range(10):
            _ = model(dummy_input)

        # Medição
        with torch.no_grad():
            for _ in range(repetitions):
                if device.type == 'cuda':
                    # Medição precisa na GPU
                    start_event = torch.cuda.Event(enable_timing=True)
                    end_event = torch.cuda.Event(enable_timing=True)
                    start_event.record()
                    _ = model(dummy_input)
                    end_event.record()
                    torch.cuda.synchronize()
                    elapsed_time_ms = start_event.elapsed_time(end_event)
                else:
                    # Medição na CPU
                    start_time = time.perf_counter()
                    _ = model(dummy_input)
                    end_time = time.perf_counter()
                    elapsed_time_ms = (end_time - start_time) * 1000

                timings.append(elapsed_time_ms)

        avg_time_ms = np.mean(timings)

        resultados_eficiencia.append({
            "Modelo": nome_modelo.replace('_', ' ').title(),
            "Parâmetros (M)": f"{total_params / 1_000_000:.2f}",
            "Tempo de Inferência (ms)": f"{avg_time_ms:.2f}"
        })

    # Salva o relatório em um arquivo de texto
    path_relatorio = os.path.join(save_dir, "relatorio_eficiencia.txt")
    with open(path_relatorio, "w") as f:
        f.write("RELATÓRIO DE EFICIÊNCIA COMPUTACIONAL\n")
        f.write("="*60 + "\n")
        f.write(f"{'Modelo':<20} | {'Parâmetros (M)':<20} | {'Tempo de Inferência (ms)':<25}\n")
        f.write("-"*60 + "\n")
        for res in resultados_eficiencia:
            f.write(f"{res['Modelo']:<20} | {res['Parâmetros (M)']:<20} | {res['Tempo de Inferência (ms)']:<25}\n")
        f.write("="*60 + "\n")
        f.write(f"\nNota: Tempo de inferência é a média de {repetitions} execuções em um dispositivo '{device.type}'.\n")

    print(f"Relatório de eficiência salvo em: {path_relatorio}")

## ORQUESTRAÇÃO




In [ ]:
# ==============================================================================
# MAIN PARA ANÁLISE DE EFICIÊNCIA
# ==============================================================================

def main():
    """Orquestra a execução da análise comparativa."""

    caminhos_resultados, relatorio_dir = configurar_caminhos(BASE_PATH, MODELOS_PARA_ANALISAR, TIPOS_DATASET)

    gerar_relatorio_eficiencia(MODELOS_PARA_ANALISAR, relatorio_dir)

main()

In [ ]:
def main():
    """Orquestra a execução da análise comparativa."""

    caminhos_resultados, relatorio_dir = configurar_caminhos(BASE_PATH, MODELOS_PARA_ANALISAR, TIPOS_DATASET)

    historicos_treinamento = carregar_dados_experimentos(caminhos_resultados, 'training')
    if historicos_treinamento:
        plotar_curvas_comparativas(historicos_treinamento, os.path.join(relatorio_dir, "comparativo_curvas_aprendizado.png"))

        # Gera os relatórios para o conjunto de VALIDAÇÃO usando a função unificada
        gerar_relatorios_unificados(historicos_treinamento, "Validation (Last Epoch)", CLASSES, relatorio_dir)

    # Etapa 3: Análise da performance final (usa dados de teste)
    resultados_teste = carregar_dados_experimentos(caminhos_resultados, 'test')
    if resultados_teste:
        # Gera os relatórios para o conjunto de TESTE usando a MESMA função unificada
        gerar_relatorios_unificados(resultados_teste, "Test", CLASSES, relatorio_dir)

In [ ]:
main()

In [ ]:
# Gera imagem com samples do dataset
import matplotlib.pyplot as plt
import cv2
import os

def gerar_figura_amostras(path_normal_orig, path_normal_wavelet, path_tumor_orig, path_tumor_wavelet, save_path):
    """
    Cria e salva uma figura 2x2 comparando imagens originais e com wavelet, com labels adicionados manualmente.
    """
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))

    img_normal_orig = cv2.imread(path_normal_orig, cv2.IMREAD_GRAYSCALE)
    img_normal_wavelet = cv2.imread(path_normal_wavelet, cv2.IMREAD_GRAYSCALE)
    img_tumor_orig = cv2.imread(path_tumor_orig, cv2.IMREAD_GRAYSCALE)
    img_tumor_wavelet = cv2.imread(path_tumor_wavelet, cv2.IMREAD_GRAYSCALE)

    axes[0, 0].imshow(img_normal_orig, cmap='gray')
    axes[0, 1].imshow(img_normal_wavelet, cmap='gray')
    axes[1, 0].imshow(img_tumor_orig, cmap='gray')
    axes[1, 1].imshow(img_tumor_wavelet, cmap='gray')

    axes[0, 0].set_title("Original", fontsize=16)
    axes[0, 1].set_title("Wavelet", fontsize=16)

    for ax in axes.flat:
        ax.axis('off')

    fig.suptitle('Dataset Samples: Original vs. Wavelet Pre-processed', fontsize=20)

    fig.text(0.06, 0.72, 'Normal', va='center', ha='center', fontsize=16, rotation='vertical')
    fig.text(0.06, 0.28, 'Tumor', va='center', ha='center', fontsize=16, rotation='vertical')

    plt.subplots_adjust(left=0.15, right=0.95, top=0.90, bottom=0.05, wspace=0.05, hspace=0.05)

    plt.savefig(save_path, dpi=300)
    plt.show()
    plt.close()
    print(f"Figura de amostras salva em: {save_path}")


path_no = ""
path_nw = ""
path_to = ""
path_tw = ""
save_path_figura = os.path.join("", 'figura_amostras_dataset.png')

gerar_figura_amostras(path_no, path_nw, path_to, path_tw, save_path_figura)
